# H08 — 浏览器查看真实设备状态

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H08 — 浏览器查看真实设备状态。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H08-http.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

现在电脑能找到 IP。给工具提供一页本地状态和一个 API，再由你添加输入校验与控制能力。

前置：H07 实际网络通；电脑 curl/浏览器。

## 本次够用的知识

HTTP server 处理请求，REST 约定资源/操作，JSON 是数据表示。handler 的返回值、HTTP 状态码与业务执行结果不同。TCP 数据可以分块，POST body 不能假定一次 recv 就收齐；处理时间有界。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 在现有网络模块上加服务 · Guided

确认 H07 的 wifi_link 只初始化一次。给本工具约定 GET /api/status、POST /api/timer 与错误 JSON 格式，先只实现 GET。
此练习限可信局域网，不直接端口映射到公网；后续设计要说明身份验证、授权和 TLS。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 启动 HTTP API · Guided

REQUIRES 增加 esp_http_server。示例主入口是最小阅读参考，持续工程只合并 status/home handler 与 server 初始化，不删除原按钮/计时模块。
获取 IP 后在浏览器打开 http://<device-ip>/，用 curl.exe（Windows）或 curl 请求 /api/status，比较 uptime_ms 是否变化。未使用 fake 网络数据。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

可选只读自动验证（在电脑终端，课程 Python 工具不要求你编写 Python）：
```text
python3 <course-root>/tools/check_device.py --url http://<device-ip>
```
检查真实 HTTP/JSON、uptime 增长与 404；这仍不能证明实物 LED 亮灭。


示例来源：`step2/lessons/H08/examples/http_status.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_http_server esp_wifi esp_event esp_netif esp_timer nvs_flash freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "wifi_link.hpp"
#include "esp_http_server.h"
#include "esp_log.h"
#include "esp_timer.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"
#include <cstdio>

static esp_err_t status(httpd_req_t* req) {
    char json[128];
    std::snprintf(json, sizeof(json), "{\"uptime_ms\":%lld,\"status\":\"ready\"}",
                  (long long)(esp_timer_get_time() / 1000));
    httpd_resp_set_type(req, "application/json");
    return httpd_resp_send(req, json, HTTPD_RESP_USE_STRLEN);
}
static esp_err_t home(httpd_req_t* req) {
    httpd_resp_set_type(req, "text/html; charset=utf-8");
    return httpd_resp_send(
        req,
        "<!doctype html><meta charset='utf-8'><style>body{background:#1e1e1e;"
        "color:#eee;font:16px/1.6 sans-serif;max-width:44rem;margin:2rem auto;"
        "padding:1rem;overflow-wrap:anywhere}a{color:#8cc8ff}</style>"
        "<h1>ESP32 Desk Tool</h1><p>真实设备状态：<a href='/api/status'>"
        "/api/status</a></p>",
        HTTPD_RESP_USE_STRLEN);
}
extern "C" void app_main() {
    ESP_ERROR_CHECK(wifi_link_start());
    while (!wifi_link_wait(pdMS_TO_TICKS(1000))) {
        ESP_LOGI("http", "waiting for IP");
    }
    httpd_handle_t server = nullptr;
    httpd_config_t config = HTTPD_DEFAULT_CONFIG();
    ESP_ERROR_CHECK(httpd_start(&server, &config));
    httpd_uri_t api{};
    api.uri = "/api/status";
    api.method = HTTP_GET;
    api.handler = status;
    ESP_ERROR_CHECK(httpd_register_uri_handler(server, &api));
    httpd_uri_t page{};
    page.uri = "/";
    page.method = HTTP_GET;
    page.handler = home;
    ESP_ERROR_CHECK(httpd_register_uri_handler(server, &page));
}


## 状态来自 Controller · Modify

把 status JSON 扩展为真实 timer_state、remaining_ms 和 firmware_version。HTTP 读取线程安全快照，不能访问正在被修改的裸字符串。
浏览器刷新时实物状态和 API 是否一致？用同一个请求记录 device-ip、响应状态和 JSON。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## URL、类型与超时 · Debug

先访问不存在的 /api/statu，记录 404；再让 Content-Type 与实际内容不一致，观察客户端行为并恢复。
POST 练习前分析 content_len=0、过大、半包、无效 JSON、未知字段：哪一种应返回 400/413/超时？不要用无界 malloc 接收任意 body。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 安全的局域网控制 · Build

实现 POST /api/timer，固定大小 body（例如上限 256 bytes），循环 recv 收齐且处理 timeout；只接受已定义的 start/cancel 和合理 seconds 范围。
valid 命令交给 Controller 所有者处理，错误输入不改变设备状态。写客户端测试：有效请求、未知操作、缺字段、类型错误、溢出、重复取消、超时。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 不用记 IP 的发现请求 · Build

用 lwIP UDP socket 实现局域网发现：只对固定版本请求响应，返回设备 ID、port、version；设置接收上界、超时与限速。
客户端先单播再选广播，记录防火墙/AP 隔离失败。与 H07 mDNS 比较取舍；来源 IP 不等于身份，UDP 无到达保证。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 做一页真正有用的面板 · Open

复用深色 #1e1e1e、可换行近白文字和明显按钮，为开始/取消/断网设计状态。布局在窄宽度检查；HTML/JSON 处理用户内容要转义。
H09 解决“刷新才能看见变化”；保留 HTTP 作为诊断基线。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 电脑 GET 获取变化的真实设备状态
- [ ] 错误 POST 不改变设备状态，响应有明确错误
- [ ] 断网/恢复与实物操作可交叉核对

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H08',
    [
        '电脑 GET 获取变化的真实设备状态',
        '错误 POST 不改变设备状态，响应有明确错误',
        '断网/恢复与实物操作可交叉核对',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H08/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/protocols/esp_http_server.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/protocols/esp_http_server.html)